# Stock Price Prediction + TensorTrade Backtest (combined)

Same pipeline as the crypto notebook, but for **stocks**: **prediction** (data, baselines, Lag+Ridge, LSTM, comparison table) and **TensorTrade backtest** (trade on Lag+Ridge signal; P&L vs buy-and-hold vs cash).

Set `ASSET` in the next cell to any yfinance stock ticker (e.g. **TSLA**, AAPL, MSFT, NVDA). Default is **TSLA** (Tesla).


In [93]:
# Config (stocks: use yfinance ticker, e.g. TSLA, AAPL, MSFT, NVDA)
ASSET = "NVDA"   # Tesla — or "AAPL", "MSFT", "NVDA", etc.
COMMISSION = 0.0005  # TensorTrade: 0.05% per trade (lower = less drag; try 0.001 or 0)
PRED_RET_THRESHOLD = 0.015   # trade when |predicted return| > this (used if USE_VAL_THRESHOLD is False)
# Asymmetric thresholds reduce sell bias (fewer sells in bull markets). Used when USE_VAL_THRESHOLD is False.
BUY_THRESHOLD = 0.008   # buy when pred_ret > this (e.g. 0.8%)
SELL_THRESHOLD = 0.025   # sell only when pred_ret < -this (e.g. 2% — fewer sells)
# Pick thresholds from validation (Section 6b): if True, backtest uses best (buy, sell) from val
USE_VAL_THRESHOLD = True
# Use LSTM predicted returns for backtest instead of Lag+Ridge (LSTM often has better Dir.Acc)
USE_LSTM_SIGNAL = True   # set True to trade on LSTM signal
# Volatility filter: only BUY/SELL when volatility_14 is in [VOL_LOW, VOL_HIGH]. Avoids high-vol regimes.
VOL_LOW = None   # e.g. 0.0 — only act when vol >= this
VOL_HIGH = 0.05  # only act when vol <= this (None = no filter). 0.05 avoids very high-vol days.


## 1. Install (run once). On Colab ignore "ERROR: ... dependency conflicts".
# !pip install -q pandas numpy yfinance pyarrow scikit-learn tensorflow matplotlib
# !pip install -q gymnasium
# !pip install -q git+https://github.com/tensortrade-org/tensortrade.git


In [94]:
!pip install -q pandas numpy yfinance pyarrow scikit-learn tensorflow matplotlib xgboost
# Optional TensorTrade (ignore Colab conflict messages):
!pip install -q gymnasium git+https://github.com/tensortrade-org/tensortrade.git


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


**Colab import errors (numpy/pandas):**  
If you see **`ValueError: numpy.dtype size changed`** or **`ImportError: cannot load module more than once per process`** when importing pandas/numpy:

1. Use **Runtime → Restart session** (or "Restart runtime").
2. After restart, run cells **from the top** (Config → Install → then Data, etc.).  
   Do **not** run the "Fix numpy/pandas" cell below unless you still see the dtype error; running pip install numpy in the same session as imports often causes the "cannot load module more than once" error.
3. If you had to run the fix cell (dtype error): run it **once**, then **Restart session** again, then run the notebook from the top and **skip** the fix cell on the second pass.

In [95]:
# Fix numpy/pandas binary mismatch (run only if you saw "numpy.dtype size changed")
#!pip install --upgrade pip -q
#!pip install --force-reinstall --no-cache-dir "numpy>=1.26" "pandas" -q
print("Done. Go to Runtime → Restart session, then re-run the notebook from the top.")

Done. Go to Runtime → Restart session, then re-run the notebook from the top.


## 2. Data and split (70/15/15)


In [96]:
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
import matplotlib.pyplot as plt

def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    n = len(y_true)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    if n > 1:
        true_dir = np.sign(np.diff(y_true))
        pred_dir = np.sign(y_pred[1:] - y_true[:-1])
        dir_acc = np.mean(true_dir == pred_dir)
    else:
        dir_acc = np.nan
    return {"mae": float(mae), "rmse": float(rmse), "directional_accuracy": float(dir_acc)}

DATA_DIR = Path("/content/data") if Path("/content").exists() else Path("./data")
DATA_DIR.mkdir(exist_ok=True)

# Cache name: stock tickers have no hyphen (e.g. TSLA), crypto use BTC-USD -> BTC_USD
cache_name = ASSET.replace("-", "_") + "_daily.parquet"
cache_path = DATA_DIR / cache_name
if cache_path.exists():
    df = pd.read_parquet(cache_path)
else:
    # Stocks: yfinance ticker (TSLA, AAPL, ...). start from 2015 for enough history.
    raw = yf.download(ASSET, start="2015-01-01", end=None, progress=False, auto_adjust=True)
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index().ffill().dropna()
    df = raw[["Close"]].copy()
    df.columns = ["price"]
    if "Volume" in raw.columns:
        df["volume"] = raw["Volume"]
    df.to_parquet(cache_path)
if "volume" not in df.columns:
    df["volume"] = 0.0
df["ret"] = (df["price"] - df["price"].shift(1)) / (df["price"].shift(1) + 1e-12)
df["volatility_14"] = df["ret"].rolling(14).std()
df["log_volume"] = np.log1p(df["volume"])

n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]
price_full = df["price"].values
log_vol_full = df["log_volume"].values
vol_full = np.nan_to_num(df["volatility_14"].values, nan=0.0)
print(df.shape, "| Train", len(train_df), "Val", len(val_df), "Test", len(test_df))


(2803, 5) | Train 1962 Val 420 Test 421


## 3. Baselines (last value, 7-day MA)


In [97]:
prices = test_df["price"].values
y_true = prices[1:]
pred_last = prices[:-1]
m_last = regression_metrics(y_true, pred_last)
window = 7
pred_ma = np.array([np.mean(prices[i - window : i]) for i in range(window, len(prices))])
y_true_ma = y_true[window - 1 :]
m_ma = regression_metrics(y_true_ma, pred_ma)
print("Last value:", m_last)
print("7-day MA:  ", m_ma)


Last value: {'mae': 3.1287462325323196, 'rmse': 4.16657153164559, 'directional_accuracy': 0.0}
7-day MA:   {'mae': 5.157903138154781, 'rmse': 6.448641867466743, 'directional_accuracy': 0.5254237288135594}


## 4. Lag+Ridge (30 lags + volume + volatility_14)


In [98]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

N_LAGS = 30
def build_lag_features(price, n_lags):
    T = len(price)
    X_list = [price[n_lags - lag : T - lag] for lag in range(1, n_lags + 1)]
    X = np.column_stack(X_list)[:-1]
    y = price[n_lags + 1 :]
    return X, y
def add_vol_volatility(X, y, start_idx, n_lags):
    n = len(y)
    idx = start_idx + n_lags
    extra = np.column_stack([log_vol_full[idx : idx + n], vol_full[idx : idx + n]])
    return np.hstack([X, extra])

X_train, y_train = build_lag_features(train_df["price"].values, N_LAGS)
X_val, y_val = build_lag_features(val_df["price"].values, N_LAGS)
X_test, y_test = build_lag_features(test_df["price"].values, N_LAGS)
X_train = add_vol_volatility(X_train, y_train, 0, N_LAGS)
X_val   = add_vol_volatility(X_val,   y_val,   train_end, N_LAGS)
X_test  = add_vol_volatility(X_test,  y_test,  val_end, N_LAGS)

n_f = X_train.shape[1]
# Try alpha in [0.1, 1, 10] if overfitting; 1.0 is a reasonable default
pipe = Pipeline([
    ("scale", ColumnTransformer([("s", StandardScaler(), list(range(n_f)))], remainder="passthrough")),
    ("ridge", Ridge(alpha=1.0)),
])
pipe.fit(X_train, y_train)
pred_lag = pipe.predict(X_test)
pred_lag_val = pipe.predict(X_val)
m_lag = regression_metrics(y_test, pred_lag)
close_val = price_full[train_end + N_LAGS : train_end + N_LAGS + len(y_val)]
actual_ret_val = (y_val - close_val) / (close_val + 1e-12)
pred_ret_val = (pred_lag_val - close_val) / (close_val + 1e-12)
print("Lag+Ridge:", m_lag)


Lag+Ridge: {'mae': 4.522052992844539, 'rmse': 5.808540628321322, 'directional_accuracy': 0.5424164524421594}


## 4b. XGBoost (same features as Ridge)

Drop-in tree model on the same lag + volume + volatility features. Often +2–5% directional accuracy vs Ridge on financial data (see MODEL_IMPROVEMENT_ASSESSMENT.md #1).

In [ ]:
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

# Same features as Ridge: scale then train XGBoost (tree models benefit from scaled inputs for consistency)
scaler_xgb = StandardScaler()
X_train_s = scaler_xgb.fit_transform(X_train)
X_val_s = scaler_xgb.transform(X_val)
X_test_s = scaler_xgb.transform(X_test)

xgb_model = xgb.XGBRegressor(
    max_depth=5,
    n_estimators=200,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train_s, y_train, verbose=False)
pred_xgb = xgb_model.predict(X_test_s)
pred_xgb_val = xgb_model.predict(X_val_s)
m_xgb = regression_metrics(y_test, pred_xgb)
pred_ret_xgb_val = (pred_xgb_val - close_val) / (close_val + 1e-12)
print("XGBoost:", m_xgb)

## 5. LSTM (predict next-day return → price)


In [99]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

# Fix random seeds so LSTM results are reproducible (otherwise Dir.Acc/MAE change every run)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

SEQ_LEN = 30
price = df["price"].values.astype(np.float64)
T = len(price)
returns = (price[1:] - price[:-1]) / (price[:-1] + 1e-12)
returns = returns.astype(np.float32)
log_vol = df["log_volume"].values.astype(np.float32)
vol = np.nan_to_num(df["volatility_14"].values, nan=0.0).astype(np.float32)

def build_seq_multifeature(ret, log_vol, vol, start, end, seq_len):
    X_list, y_list = [], []
    for i in range(start, min(end, len(ret) - 1)):
        if i >= seq_len:
            X_list.append(np.column_stack([ret[i - seq_len : i], log_vol[i - seq_len : i], vol[i - seq_len : i]]))
            y_list.append(ret[i])
    if not X_list:
        return np.zeros((0, seq_len, 3), dtype=np.float32), np.array([], dtype=np.float32)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

X_tr, y_tr = build_seq_multifeature(returns, log_vol, vol, SEQ_LEN, train_end, SEQ_LEN)
X_va, y_va = build_seq_multifeature(returns, log_vol, vol, train_end, val_end, SEQ_LEN)
X_te, y_te = build_seq_multifeature(returns, log_vol, vol, val_end, T - 1, SEQ_LEN)
scaler = StandardScaler()
n_tr, seq_len, n_feat = X_tr.shape
scaler.fit(X_tr.reshape(-1, n_feat))
X_tr = scaler.transform(X_tr.reshape(-1, n_feat)).reshape(-1, seq_len, n_feat).astype(np.float32)
X_va = scaler.transform(X_va.reshape(-1, n_feat)).reshape(-1, seq_len, n_feat).astype(np.float32)
X_te = scaler.transform(X_te.reshape(-1, n_feat)).reshape(-1, seq_len, n_feat).astype(np.float32)

model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, n_feat)),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(16),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=30, batch_size=32, verbose=0)
pred_ret = model.predict(X_te, verbose=0).ravel()
pred_ret_val_lstm = model.predict(X_va, verbose=0).ravel()  # for validation threshold sweep when USE_LSTM_SIGNAL
test_start_price_idx = val_end
price_prev = price[test_start_price_idx : test_start_price_idx + len(pred_ret)]
pred_lstm_price = price_prev * (1 + pred_ret)
y_true_price = price[test_start_price_idx + 1 : test_start_price_idx + 1 + len(pred_ret)]
m_lstm = regression_metrics(y_true_price, pred_lstm_price)
print("LSTM:", m_lstm)


LSTM: {'mae': 3.286589956563311, 'rmse': 4.412182342105894, 'directional_accuracy': 0.5287081339712919}


## 6. Comparison table


In [100]:
rows = [
    ["Last value", m_last["mae"], m_last["rmse"], m_last["directional_accuracy"]],
    ["7-day MA", m_ma["mae"], m_ma["rmse"], m_ma["directional_accuracy"]],
    ["Lag+Ridge", m_lag["mae"], m_lag["rmse"], m_lag["directional_accuracy"]],
    ["XGBoost", m_xgb["mae"], m_xgb["rmse"], m_xgb["directional_accuracy"]],
    ["LSTM", m_lstm["mae"], m_lstm["rmse"], m_lstm["directional_accuracy"]],
]
print(pd.DataFrame(rows, columns=["Model", "MAE", "RMSE", "Dir.Acc"]).to_string(index=False))


     Model      MAE     RMSE  Dir.Acc
Last value 3.128746 4.166572 0.000000
  7-day MA 5.157903 6.448642 0.525424
 Lag+Ridge 4.522053 5.808541 0.542416
      LSTM 3.286590 4.412182 0.528708


## 6b. Pick thresholds from validation (symmetric or asymmetric)

Simulate the threshold policy on **validation**: try (buy_thresh, sell_thresh) pairs. **Asymmetric** (sell_thresh > buy_thresh) reduces sell bias. When `USE_LSTM_SIGNAL` is True we use LSTM validation predictions; otherwise Lag+Ridge. Best pair is used in backtest when `USE_VAL_THRESHOLD = True`.

In [101]:
# Use LSTM or Lag+Ridge validation predictions for threshold search
signal_val = pred_ret_val_lstm if USE_LSTM_SIGNAL else pred_ret_val
# Align lengths: LSTM val can have different length than actual_ret_val (from Lag+Ridge val period)
n_val = min(len(signal_val), len(actual_ret_val))
signal_val = np.asarray(signal_val).ravel()[:n_val]
actual_ret_val_aligned = actual_ret_val[:n_val]
# Asymmetric grid: (buy_thresh, sell_thresh). sell > buy = fewer sells.
candidates_buy = [0.005, 0.008, 0.01, 0.015]
candidates_sell = [0.01, 0.015, 0.02, 0.025, 0.03]
val_results = []
for buy_t in candidates_buy:
    for sell_t in candidates_sell:
        if sell_t < buy_t:
            continue  # require sell_thresh >= buy_thresh
        position = 0
        cum_ret = 0.0
        for i in range(n_val):
            if signal_val[i] > buy_t:
                position = 1
            elif signal_val[i] < -sell_t:
                position = 0
            cum_ret += position * actual_ret_val_aligned[i]
        val_results.append((buy_t, sell_t, cum_ret))
best_buy, best_sell, best_cret = max(val_results, key=lambda x: x[2])
best_buy_threshold_from_val = best_buy
best_sell_threshold_from_val = best_sell
best_threshold_from_val = best_buy  # backward compat for symmetric use
print("Validation: (buy_thresh, sell_thresh) → cumulative return (hypothetical)")
for buy_t, sell_t, cret in sorted(val_results, key=lambda x: -x[2])[:10]:
    mark = " ← best" if (buy_t, sell_t) == (best_buy, best_sell) else ""
    print(f"  ({buy_t:.3f}, {sell_t:.3f})  →  {cret:+.4f}{mark}")
print(f"Use buy={best_buy_threshold_from_val}, sell={best_sell_threshold_from_val} in backtest when USE_VAL_THRESHOLD is True.")

Validation: (buy_thresh, sell_thresh) → cumulative return (hypothetical)
  (0.005, 0.025)  →  +2.3596 ← best
  (0.005, 0.030)  →  +2.3596
  (0.008, 0.025)  →  +2.3596
  (0.008, 0.030)  →  +2.3596
  (0.010, 0.025)  →  +1.9570
  (0.010, 0.030)  →  +1.9570
  (0.005, 0.015)  →  +1.7818
  (0.005, 0.020)  →  +1.7260
  (0.008, 0.015)  →  +1.6492
  (0.008, 0.020)  →  +1.5935
Use buy=0.005, sell=0.025 in backtest when USE_VAL_THRESHOLD is True.


## 7. TensorTrade backtest (Lag+Ridge or LSTM signal)

Trading signal: **Lag+Ridge** or **LSTM** (set `USE_LSTM_SIGNAL` in config). At each step: BUY if pred_ret > buy_thresh, SELL if pred_ret < -sell_thresh, else HOLD. Thresholds from **validation** (Section 6b) when `USE_VAL_THRESHOLD = True`, else from config (asymmetric defaults). **Volatility filter:** only trade when volatility_14 ≤ `VOL_HIGH` (and ≥ VOL_LOW if set). TensorTrade simulates execution and commission.


In [102]:
# Build backtest window: signal from LSTM or Lag+Ridge
start_idx = val_end + N_LAGS
close_arr = price_full[start_idx : start_idx + len(y_test)]
vol_arr = vol_full[start_idx : start_idx + len(y_test)]
log_vol_arr = log_vol_full[start_idx : start_idx + len(y_test)]
if USE_LSTM_SIGNAL:
    # Align LSTM test predictions (last len(y_test)) to same backtest window
    pred_ret_arr = np.asarray(pred_ret).ravel()[-len(y_test):].copy()
else:
    pred_ret_arr = (pred_lag - close_arr) / (close_arr + 1e-12)
close_list = close_arr.tolist()
volume_list = np.expm1(log_vol_arr).tolist()
vol_list = vol_arr.tolist()
pred_ret_list = pred_ret_arr.tolist()
T = len(close_list)
print("Backtest length:", T, "steps | Signal:", "LSTM" if USE_LSTM_SIGNAL else "Lag+Ridge", "| VOL_HIGH:", VOL_HIGH)


Backtest length: 390 steps | Signal: LSTM | VOL_HIGH: 0.05


In [103]:
try:
    from tensortrade.feed.core import DataFeed, Stream
    from tensortrade.oms.exchanges import Exchange, ExchangeOptions
    from tensortrade.oms import instruments as tt_instruments
    from tensortrade.oms.instruments import Instrument
    from tensortrade.oms.services.execution.simulated import execute_order
    from tensortrade.oms.wallets import Wallet, Portfolio
    from tensortrade.env.default.actions import BSH
    from tensortrade.env.default.rewards import PBR
    import tensortrade.env.default as default
    # Stock tickers are plain (TSLA, AAPL); crypto use BTC-USD -> take first part
    _sym = ASSET.split("-")[0] if "-" in ASSET else ASSET
    ASSET_INSTRUMENT = getattr(tt_instruments, _sym, None)
    if ASSET_INSTRUMENT is None:
        # TensorTrade has no built-in for stocks: create custom instrument (precision=2 for price)
        ASSET_INSTRUMENT = Instrument(_sym, 2, _sym)
    TENSORTRADE_AVAILABLE = True
except ImportError as e:
    TENSORTRADE_AVAILABLE = False
    print("TensorTrade not installed. Run: pip install gymnasium && pip install git+https://github.com/tensortrade-org/tensortrade.git"); print("Error:", e)


In [104]:
if TENSORTRADE_AVAILABLE:
    _pair = f"USD-{_sym}"
    price = Stream.source(close_list, dtype="float").rename(_pair)
    exchange_options = ExchangeOptions(commission=COMMISSION)
    exchange = Exchange("exchange", service=execute_order, options=exchange_options)(price)
    initial_cash = 10000.0
    cash = Wallet(exchange, initial_cash * tt_instruments.USD)
    asset = Wallet(exchange, 0 * ASSET_INSTRUMENT)
    portfolio = Portfolio(tt_instruments.USD, [cash, asset])
    features = [
        Stream.source(close_list, dtype="float").rename("close"),
        Stream.source(volume_list, dtype="float").rename("volume"),
        Stream.source(vol_list, dtype="float").rename("volatility_14"),
        Stream.source(pred_ret_list, dtype="float").rename("model_pred_ret"),
    ]
    feed = DataFeed(features)
    feed.compile()
    reward_scheme = PBR(price=price)
    action_scheme = BSH(cash=cash, asset=asset).attach(reward_scheme)
    env = default.create(feed=feed, portfolio=portfolio, action_scheme=action_scheme, reward_scheme=reward_scheme, window_size=5, max_allowed_loss=0.5)
    print("Environment created. Commission:", COMMISSION)


Environment created. Commission: 0.0005


In [105]:
if TENSORTRADE_AVAILABLE:
    obs, info = env.reset()
    done = truncated = False
    step = 0
    total_reward = 0.0
    n_buys = n_sells = 0
    if USE_VAL_THRESHOLD:
        buy_thresh = best_buy_threshold_from_val
        sell_thresh = best_sell_threshold_from_val
    else:
        buy_thresh = BUY_THRESHOLD if BUY_THRESHOLD is not None else PRED_RET_THRESHOLD
        sell_thresh = SELL_THRESHOLD if SELL_THRESHOLD is not None else PRED_RET_THRESHOLD
    while not (done or truncated) and step < T:
        pred_ret = pred_ret_list[step]
        vol = vol_arr[step]
        if VOL_LOW is not None and vol < VOL_LOW:
            action = 2
        elif VOL_HIGH is not None and vol > VOL_HIGH:
            action = 2
        elif pred_ret > buy_thresh:
            action = 0
            n_buys += 1
        elif pred_ret < -sell_thresh:
            action = 1
            n_sells += 1
        else:
            action = 2
        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward) if np.isscalar(reward) else float(np.asarray(reward).item())
        step += 1
    final_worth = portfolio.net_worth
    pnl = final_worth - initial_cash
    pnl_pct = 100 * (pnl / initial_cash)
    buy_hold_return = (close_list[-1] - close_list[0]) / (close_list[0] + 1e-12)
    buy_hold_pnl = initial_cash * buy_hold_return
    print("--- Model-threshold policy ---")
    print(f"  Steps: {step}  Buys: {n_buys}  Sells: {n_sells}  Total reward: {total_reward:.2f}")
    print(f"  Final: ${final_worth:,.2f}  P&L: ${pnl:+,.2f} ({pnl_pct:+.2f}%)")
    print("--- Buy-and-hold ---")
    print(f"  P&L: ${buy_hold_pnl:+,.2f} ({100*buy_hold_return:+.2f}%)")
    print("--- Cash (0%) ---")
    print(f"  P&L: $0.00 (0.00%)")
    if pnl > buy_hold_pnl and pnl > 0:
        print("Model beat both buy-and-hold and cash.")
    elif pnl > buy_hold_pnl:
        print("Model lost less than buy-and-hold (held more cash). Versus cash, model lost money.")
    elif pnl > 0:
        print("Model beat cash but lost to buy-and-hold.")
    else:
        print("Model lost to both.")


--- Model-threshold policy ---
  Steps: 389  Buys: 296  Sells: 0  Total reward: 41.94
  Final: $16,296.79  P&L: $+6,296.79 (+62.97%)
--- Buy-and-hold ---
  P&L: $+9,206.82 (+92.07%)
--- Cash (0%) ---
  P&L: $0.00 (0.00%)
Model beat cash but lost to buy-and-hold.


## 8. Improving results — what to try

- **Validation thresholds:** Keep `USE_VAL_THRESHOLD = True` to use the best (buy, sell) pair from Section 6b (asymmetric reduces sell bias).
- **LSTM signal:** Set `USE_LSTM_SIGNAL = True` to backtest on LSTM predictions (often better Dir.Acc than Lag+Ridge).
- **Volatility filter:** `VOL_HIGH = 0.05` (default) skips trading on very high-vol days; set to `None` to disable. `VOL_LOW` can avoid dead markets.
- **Asymmetric when not using val:** With `USE_VAL_THRESHOLD = False`, defaults are `BUY_THRESHOLD = 0.008`, `SELL_THRESHOLD = 0.02` (fewer sells).
- **Ridge alpha:** In Section 4 try `alpha=0.1` or `10` if val/test metrics suggest over/underfitting.
- **Threshold sweep** below: explore symmetric thresholds on test (for asymmetric, use Section 6b + backtest).

In [106]:
# Threshold sweep: try several thresholds and report P&L (run after the backtest above)
if TENSORTRADE_AVAILABLE:
    results = []
    for thresh in [0.005, 0.01, 0.015, 0.02]:
        cl = close_arr.tolist()
        vl = np.expm1(log_vol_arr).tolist()
        v14 = vol_arr.tolist()
        pr = pred_ret_arr.tolist()
        price_s = Stream.source(cl, dtype="float").rename(f"USD-{_sym}")
        ex = Exchange("ex", service=execute_order, options=ExchangeOptions(commission=COMMISSION))(price_s)
        c = Wallet(ex, 10000.0 * tt_instruments.USD)
        a = Wallet(ex, 0 * ASSET_INSTRUMENT)
        port = Portfolio(tt_instruments.USD, [c, a])
        feed = DataFeed([Stream.source(cl, dtype="float").rename("close"), Stream.source(vl, dtype="float").rename("vol"), Stream.source(v14, dtype="float").rename("v14"), Stream.source(pr, dtype="float").rename("pred_ret")])
        feed.compile()
        rwd = PBR(price=price_s)
        act = BSH(cash=c, asset=a).attach(rwd)
        env2 = default.create(feed=feed, portfolio=port, action_scheme=act, reward_scheme=rwd, window_size=5, max_allowed_loss=0.5)
        obs, _ = env2.reset()
        done, truncated, step = False, False, 0
        while not (done or truncated) and step < len(cl):
            action = 0 if pr[step] > thresh else (1 if pr[step] < -thresh else 2)
            obs, r, done, truncated, _ = env2.step(action)
            step += 1
        pnl_pct = 100 * (port.net_worth - 10000.0) / 10000.0
        results.append((thresh, pnl_pct, step))
    print("Threshold | P&L %  | Steps")
    for thresh, pnl_pct, steps in results:
        print(f"  {thresh:.3f}   | {pnl_pct:+.2f}% | {steps}")
    best = max(results, key=lambda x: x[1])
    print(f"Best in sweep: threshold {best[0]:.3f} → P&L {best[1]:+.2f}%")

Threshold | P&L %  | Steps
  0.005   | +25.89% | 389
  0.010   | +14.64% | 389
  0.015   | +29.00% | 389
  0.020   | +42.94% | 389
Best in sweep: threshold 0.020 → P&L +42.94%
